# Selection audit — what is knowable before any retrieval

The cell fill picks *which* queries the dataset contains. Whether those queries
produce clean route labels needs corpora, embeddings and retrieval — none of
which this notebook touches.

Everything below is a property of the **selection itself**, computable from
`cell_selection.parquet` and the feature catalog. Each section names the
question it answers and, where it exists, the number the old slice fill scored.

In [ ]:
import sys
sys.path.insert(0, ".")

import pandas as pd

from composition.cells import CELLS
from composition.recipe import Recipe

CATALOG = "data/feature_table/catalog.parquet"
NEW = "data/composition/cell_selection.parquet"
OLD = "data/composition/selection.parquet"
REPORT = "data/composition/cell_report.parquet"

key = lambda f: f["dataset"] + "|" + f["query_id"].astype(str)
catalog = pd.read_parquet(CATALOG).assign(key=lambda d: key(d))
new = pd.read_parquet(NEW).assign(key=lambda d: key(d))
old = pd.read_parquet(OLD).assign(key=lambda d: key(d))
report = pd.read_parquet(REPORT)
recipe = Recipe()

print(f"catalog {len(catalog):,} rows / {catalog['dataset'].nunique()} lanes")
print(f"new selection {len(new):,} rows / {new['dataset'].nunique()} lanes / {new['cell'].nunique()} cells")
print(f"n_per_route = {recipe.n_per_route}  ->  decisive target {len(CELLS) * recipe.n_per_route * 2:,}")

In [ ]:
new['cell'].unique()

## 1. Did the lane confound break?

The defect the redesign exists to fix: route outcome was a property of *which
corpus* a query came from. A cell drawn mostly from one corpus teaches that
corpus, not the archetype. Under the old fill, 12 of 23 populated cells drew
≥80% of their rows from a single lane.

One caveat this cannot answer: spread is necessary, not sufficient. Whether
decorrelation actually changes what the router learns needs labels.

In [ ]:
def top_lane_share(frame, group, floor=20):
    out = {}
    for name, g in frame.groupby(group):
        if len(g) < floor:
            continue
        counts = g["dataset"].value_counts()
        out[name] = counts.iloc[0] / counts.sum()
    return pd.Series(out, name="top_lane_share")

new_share = top_lane_share(new, "cell")
old_share = top_lane_share(old, "slice")

summary = pd.DataFrame({
    "groups": [len(new_share), len(old_share)],
    "mean": [new_share.mean(), old_share.mean()],
    "median": [new_share.median(), old_share.median()],
    "worst": [new_share.max(), old_share.max()],
    ">=80% one lane": [(new_share >= 0.8).sum(), (old_share >= 0.8).sum()],
}, index=["new (per cell)", "old (per slice)"])
display(summary.round(2))

new_share.sort_values(ascending=False).head(10).to_frame().style.format("{:.0%}")

## 2. Which register did we end up with?

The old selection was criticised for being natural-language-heavy, so
auto-fusion had nothing to bite on and the canonical sparse archetypes were
thin. Wave 2 added WebFAQ, GooAQ and CLERC — all natural-question registers —
so this is where the acquisition could have pulled the wrong way.

In [ ]:
identifier_columns = [c for c in catalog.columns if c.startswith("structured_identifiers.")]
catalog["identifier_spans"] = catalog[identifier_columns].sum(axis=1)

def register(keys, label):
    j = catalog[catalog["key"].isin(set(keys))]
    return {
        "selection": label,
        "rows_in_catalog": len(j),
        "median_words": j["length.length_words"].median(),
        "short_<=6w": (j["length.length_words"] <= 6).mean(),
        "nl_share": j["natural_language_signal.natural_language_share"].mean(),
        "any_identifier": (j["identifier_spans"] > 0).mean(),
        "lanes": j["dataset"].nunique(),
    }

pd.DataFrame([register(old["key"], "old d32"), register(new["key"], "new cells")]).set_index("selection").round(2)

## 3. Which archetypes actually got supply?

`natural_rows` is what the cell could draw from; the draw is `2 * n_per_route`.
A cell below its draw is short, and one flagged `lane_share_infeasible` could
not reach the target lane share however it drew — its lanes are too thin.

In [ ]:
draw = recipe.n_per_route * 2
view = report[[
    "cell", "lanes", "natural_rows", "top_share", "lane_share_cap",
    "lane_share_achieved", "lane_share_infeasible",
    "reused_dense", "reused_sparse", "labels_queued", "augmentation_rows",
]].copy()
view["short_of_draw"] = view["natural_rows"] < draw

print(f"draw per cell         {draw}")
print(f"cells short of draw   {view['short_of_draw'].sum()} of {len(view)}")
print(f"lane_share_infeasible {int(view['lane_share_infeasible'].sum())} of {len(view)}")
print(f"augmentation rows     {int(view['augmentation_rows'].sum())}")
view.sort_values("natural_rows").head(12)

## 4. What did each cell actually match?

The aggregate numbers cannot tell you whether a predicate caught what it meant
to. This prints real queries per cell alongside the measured band values that
made each row match, so a cell matching noise is visible rather than inferred.

Two were already found this way: `uri_in_query` matches long questions that
merely *mention* a URL (its predicate has no length bound), and
`opaque_token_any_domain` matched `"how to get 277 V AC ?"` because the
`http_status_code` bank fired on `277`.

In [ ]:
by_name = {c.name: c for c in CELLS}
measured = new.merge(catalog.drop(columns=["dataset", "query_id"]), on="key", how="left")


def samples(cell_name, n=4, width=90):
    """Queries a cell claimed, with the band values that made them match."""
    cell = by_name[cell_name]
    bands = list(cell.predicate) + list(cell.any_of)
    rows = measured[measured["cell"] == cell_name].head(n)
    clip = lambda t: t[:width] + ("…" if len(t) > width else "")
    return pd.DataFrame([
        {
            "lane": row["dataset"],
            "stage": row["stage"],
            "query": clip(" ".join(str(row["query"]).split())),
            **{band.member: row.get(band.column) for band in bands},
        }
        for _, row in rows.iterrows()
    ])


samples("keyword_telegram_short", width=500)

In [ ]:
# thinnest cells first — these are the augmentation-critical ones
for name in report.nsmallest(6, "natural_rows")["cell"]:
    print(f"=== {name}  predicts {'+'.join(by_name[name].predicts)}")
    frame = samples(name, n=3)
    print(frame.to_string(index=False) if len(frame) else "  (no rows)")
    print()

In [ ]:
# and the largest, where a loose predicate would quietly dominate the dataset
for name in report.nlargest(4, "labels_queued")["cell"]:
    print(f"=== {name}")
    print(samples(name, n=3).to_string(index=False))
    print()

### Rationale vs reality

Each cell carries the mechanism its author claimed. Read it against the samples
above: a rationale that does not describe what actually matched is a predicate
bug, not a labelling question, and it is fixable before any retrieval runs.

In [ ]:
pd.set_option("display.max_colwidth", 160)
pd.DataFrame([
    {"cell": c.name, "predicts": "+".join(c.predicts), "rationale": c.rationale}
    for c in CELLS
]).set_index("cell")

## 5. Is the queue supply-bound or quota-bound?

`labels_wanted` is what the quota needs; `labels_queued` is what the cells could
actually offer. A large gap means the ceiling is per-cell supply under the lane
cap — which `recommended_sample` on the big boxes controls, not `n_per_route`.

In [ ]:
wanted, queued = int(report["labels_wanted"].sum()), int(report["labels_queued"].sum())
print(f"labels_wanted  {wanted:,}")
print(f"labels_queued  {queued:,}  ({queued / wanted:.0%} of wanted)")
print(f"reused free    {int((new['stage'] == 'reused').sum()):,}")
print()
print("box sample caps vs what exists upstream:")
pd.DataFrame([
    {"box": "clerc", "sampled": 50_000, "available": 327_414},
    {"box": "gooaq", "sampled": 50_000, "available": 3_031_709},
    {"box": "webfaq-eng", "sampled": 50_000, "available": 5_278_725},
    {"box": "orcas", "sampled": 100_000, "available": 10_405_342},
]).assign(share=lambda d: (d["sampled"] / d["available"]).map("{:.1%}".format)).set_index("box")

## 6. Near-duplicate leakage

~5.5% of the old selection were cos>0.95 pairs, and the train/eval split was not
near-duplicate aware — so within-lane numbers may have been inflated by an
unknown amount. Measuring it now lets the first split be near-dup aware instead
of discovering it afterwards.

This is an **auxiliary** embedding of query strings via sentence-transformers —
unrelated to the fastembed retrieval stack, and it does not need any corpus.
Minutes, not hours. Set `RUN_NEAR_DUP = True` to execute.

In [ ]:
RUN_NEAR_DUP = True

if RUN_NEAR_DUP:
    import numpy as np
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("all-MiniLM-L6-v2")
    texts = new["query"].fillna("").tolist()
    vectors = model.encode(texts, batch_size=256, show_progress_bar=True,
                           normalize_embeddings=True)
    # blocked cosine so 32K x 32K never materialises
    threshold, pairs = 0.95, 0
    for start in range(0, len(vectors), 2048):
        block = vectors[start:start + 2048] @ vectors.T
        block[np.arange(len(block)), np.arange(start, start + len(block))] = 0.0
        pairs += int((block > threshold).sum())
    print(f"cos>{threshold} pairs: {pairs // 2:,} over {len(vectors):,} queries")
else:
    print("skipped — set RUN_NEAR_DUP = True")